In [1]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00


# ASM

In [2]:
# import torch
# import torch.nn as nn
# import pandas as pd
# import sacrebleu
# import glob
# import os
# import random
# import re
# import numpy as np
# from tqdm import tqdm
# from transformers import AutoModel, AutoTokenizer, MT5ForConditionalGeneration
# from transformers.modeling_outputs import BaseModelOutput
# import torch.nn.functional as F
# from huggingface_hub import login
# from kaggle_secrets import UserSecretsClient

# # Fetch the secret token
# user_secrets = UserSecretsClient()
# hf_token = user_secrets.get_secret("HF_TOKEN")

# # Login automatically
# login(token=hf_token, add_to_git_credential=False)

# # --- CONFIGURATION ---
# TARGET_LANG = "as"  # The Decoder Language we are testing
# DECODER_WEIGHTS = "/kaggle/input/models/drneelimasingh/decoder-asm-beng/pytorch/default/1/decoder_asm_Beng.pt"
# DAE_PATH = '/kaggle/input/models/vedantm23/dae-model-epoch-7/pytorch/finalmodel/1/dae_model_final.pt'

# # Shared Paths (Universal Encoder)
# SHARED_ENC_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_context_encoder.pt"
# SHARED_PROJ_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_projector.pt"

# BATCH_SIZE = 16

# # --- HELPER: RESTORE MASKED NUMBERS ---
# def restore_masked_numbers(source_text, predicted_text):
#     """
#     Extracts numbers/dates from the source text and replaces placeholder tags 
#     (like <আইডি১> or <ID1>) in the predicted text.
#     """
#     source_numbers = re.findall(r'[\d\.]+\d', source_text)
#     restored_text = predicted_text
    
#     for num in source_numbers:
#         restored_text = re.sub(r'<[^>]+>', num, restored_text, count=1)
        
#     restored_text = re.sub(r'<[^>]+>', '', restored_text)
#     return restored_text

# # --- HELPER: IndicBERTScore ---
# def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
#     scores = []
#     for i in range(0, len(predictions), 32):
#         batch_preds = predictions[i : i + 32]
#         batch_refs = references[i : i + 32]
        
#         with torch.no_grad():
#             input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
#             input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
#             out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
#             out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
#             out_preds = F.normalize(out_preds, p=2, dim=1)
#             out_refs = F.normalize(out_refs, p=2, dim=1)
            
#             sims = (out_preds * out_refs).sum(dim=1)
#             scores.extend(sims.cpu().tolist())

#     return sum(scores) / len(scores) if scores else 0

# # --- 1. PYTORCH ARCHITECTURE ---
# class DAE(nn.Module):
#     def __init__(self, d_in=768, d_latent=256):
#         super().__init__()
#         self.encoder = nn.Sequential(
#             nn.Linear(d_in, 512), nn.ReLU(),
#             nn.Linear(512, d_latent)
#         )
#         self.decoder = nn.Sequential(
#             nn.Linear(d_latent, 512), nn.ReLU(),
#             nn.Linear(512, d_in)
#         )

#     def forward(self, x):
#         z = self.encoder(x)
#         x_hat = self.decoder(z)
#         return x_hat, z

# class ContextualEncoder(nn.Module):
#     def __init__(self, d_model=256, nhead=4, num_layers=4):
#         super().__init__()
#         encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
#         self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
#     def forward(self, x, mask=None):
#         if mask is not None:
#             padding_mask = (mask == 0) 
#             return self.transformer_encoder(x, src_key_padding_mask=padding_mask)
#         return self.transformer_encoder(x)

# class CustomMTPipeline(nn.Module):
#     def __init__(self, dae_path, mt5_model_name="google/mt5-small"):
#         super().__init__()

#         self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#         # IndicBERT
#         self.indic = AutoModel.from_pretrained("ai4bharat/indic-bert")

#         # DAE
#         self.dae = DAE(d_in=768, d_latent=256)

#         checkpoint = torch.load(dae_path, map_location=self.device)
#         if 'model_state_dict' in checkpoint:
#             self.dae.load_state_dict(checkpoint['model_state_dict'])
#         else:
#             self.dae.load_state_dict(checkpoint)

#         for param in self.dae.parameters():
#             param.requires_grad = False

#         # Context Encoder + Projector
#         self.context_encoder = ContextualEncoder(d_model=256, nhead=4, num_layers=4)
#         self.projector = nn.Linear(256, 512)

#         # ✅ LOAD MT5 CORRECTLY
#         self.t5 = MT5ForConditionalGeneration.from_pretrained(mt5_model_name)

#         # IMPORTANT when loading custom decoder weights
#         self.t5.config.tie_word_embeddings = False

#         # Remove encoder (IndicBERT is acting as encoder)
#         del self.t5.encoder

#     def forward(self, src_ids, src_mask, tgt_ids):

#         indic_out = self.indic(src_ids, attention_mask=src_mask).last_hidden_state
#         _, latent_z = self.dae(indic_out)

#         context_out = self.context_encoder(latent_z, mask=src_mask)
#         decoder_input_embeds = self.projector(context_out)

#         encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)

#         out = self.t5(
#             encoder_outputs=encoder_outputs_wrapped,
#             labels=tgt_ids,
#             attention_mask=src_mask
#         )

#         return out.loss


# # --- 2. SETUP PIPELINE & EVAL MODELS ---
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# indic_tok = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
# t5_tok = AutoTokenizer.from_pretrained("google/mt5-small", use_fast=False)

# print("Loading Custom Pipeline Weights...")
# model = CustomMTPipeline(DAE_PATH, mt5_model_name="google/mt5-small").to(device)
# # We still load the weights so the model object initializes properly, even though we ablate the context encoder
# model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
# model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
# model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
# model.eval()

# print("Loading Global Metric Evaluators...")
# eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
# eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
# eval_model.eval()

# # --- 3. FIND UNIQUE SOURCE FILES ---
# print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
# file_pattern = f"*_to_{TARGET_LANG}.csv"
# all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

# if not all_files:
#     print("No files found! Check your directory.")
#     exit()

# # --- 4. MAIN EVALUATION LOOP (ABLATION MODE) ---
# results = []
# processed_languages = set() 

# for file_path in all_files:
#     try:
#         header_df = pd.read_csv(file_path, nrows=0)
#         cols = header_df.columns.tolist()
        
#         if TARGET_LANG not in cols:
#             continue 
            
#         src_col = [c for c in cols if c != TARGET_LANG][0]
        
#         if src_col in processed_languages:
#             continue 
        
#         processed_languages.add(src_col)
#         print(f"\n[ABLATION] Testing Pair: {src_col} -> {TARGET_LANG} (Bypassing Contextual Encoder)")
        
#         df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
#         sample_df = df.dropna().astype(str)
        
#         src_texts = sample_df[src_col].tolist()
#         ref_texts = sample_df[TARGET_LANG].tolist()
        
#         predictions = []
#         for i in range(0, len(src_texts), BATCH_SIZE):
#             batch_src = src_texts[i : i + BATCH_SIZE]
            
#             inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
#             with torch.no_grad():
#                 # 1. IndicBERT
#                 indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
#                 # 2. DAE Tokenwise Extraction
#                 _, latent_z = model.dae(indic_out)
                
#                 # =========================================================
#                 # 🚨 ABLATION OVERRIDE: BYPASS THE CONTEXTUAL ENCODER 🚨
#                 # Instead of passing latent_z to context_encoder, we send it 
#                 # directly to the projector to test translation without sentence-level context.
#                 # =========================================================
#                 decoder_input_embeds = model.projector(latent_z)
                
#                 encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
#                 # 3. Target Generation
#                 gen_ids = model.t5.generate(
#                     encoder_outputs=encoder_outputs_wrapped,
#                     attention_mask=inputs.attention_mask,
#                     max_length=64,
#                     num_beams=4,
#                     do_sample=False
#                 )
                
#             batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            
#             # Post-processing to fix placeholder tags
#             for j in range(len(batch_preds)):
#                 batch_preds[j] = restore_masked_numbers(batch_src[j], batch_preds[j])
                
#             predictions.extend(batch_preds)
            
#         # 🕵️ DIAGNOSTIC BLOCK
#         print("\n" + "="*70)
#         print(f"🕵️ ABLATION SAMPLES FOR ({src_col} -> {TARGET_LANG}) 🕵️")
#         print("="*70)

#         sample_indices = random.sample(range(len(predictions)), min(5, len(predictions)))

#         for idx, rand_i in enumerate(sample_indices):
#             print(f"\n--- Sample {idx+1} ---")
#             print(f"Source ({src_col}):      {src_texts[rand_i]}")
#             print(f"Reference ({TARGET_LANG}):   {ref_texts[rand_i]}")
#             print(f"Prediction ({TARGET_LANG}):  {predictions[rand_i]}")

#         print("\n" + "="*70)
            
#         # Calculate Metrics
#         bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
#         chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
#         bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
#         print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
#         results.append({
#             "Source Language": src_col,
#             "BLEU": bleu.score,
#             "ChrF++": chrf.score,
#             "IndicBERT": bert
#         })
        
#     except Exception as e:
#         print(f"Error processing {os.path.basename(file_path)}: {e}")

# # --- 5. FINAL REPORT ---
# if results:
#     results_df = pd.DataFrame(results)
    
#     print("\n" + "="*70)
#     print(f"FINAL AGGREGATE RESULTS FOR ABLATION (No Contextual Encoder): {TARGET_LANG}")
#     print("="*70)
#     print(results_df.round(2).to_string(index=False))
#     print("-" * 70)
    
#     avg_bleu = results_df["BLEU"].mean()
#     avg_chrf = results_df["ChrF++"].mean()
#     avg_bert = results_df["IndicBERT"].mean()
    
#     print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
#     print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
#     print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
#     print("="*70)
    
#     # Save with a specific ablation filename
#     results_df.to_csv(f"ablation_no_context_results_{TARGET_LANG}.csv", index=False)
#     print(f"✓ Ablation results saved to ablation_no_context_results_{TARGET_LANG}.csv")
# else:
#     print("No results generated.")

# BENG

In [3]:
# import torch
# import torch.nn as nn
# import pandas as pd
# import sacrebleu
# import glob
# import os
# import random
# import re
# import numpy as np
# from tqdm import tqdm
# from transformers import AutoModel, AutoTokenizer, MT5ForConditionalGeneration
# from transformers.modeling_outputs import BaseModelOutput
# import torch.nn.functional as F
# from huggingface_hub import login
# from kaggle_secrets import UserSecretsClient

# # Fetch the secret token
# user_secrets = UserSecretsClient()
# hf_token = user_secrets.get_secret("HF_TOKEN")

# # Login automatically
# login(token=hf_token, add_to_git_credential=False)

# # --- CONFIGURATION ---
# TARGET_LANG = "bn"  # The Decoder Language we are testing
# DECODER_WEIGHTS = "/kaggle/input/models/waibhavjha/decoder-bengali/pytorch/default/1/decoder_ben_Beng.pt"
# DAE_PATH = '/kaggle/input/models/vedantm23/dae-model-epoch-7/pytorch/finalmodel/1/dae_model_final.pt'

# # Shared Paths (Universal Encoder)
# SHARED_ENC_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_context_encoder.pt"
# SHARED_PROJ_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_projector.pt"

# BATCH_SIZE = 16

# # --- HELPER: RESTORE MASKED NUMBERS ---
# def restore_masked_numbers(source_text, predicted_text):
#     """
#     Extracts numbers/dates from the source text and replaces placeholder tags 
#     (like <আইডি১> or <ID1>) in the predicted text.
#     """
#     source_numbers = re.findall(r'[\d\.]+\d', source_text)
#     restored_text = predicted_text
    
#     for num in source_numbers:
#         restored_text = re.sub(r'<[^>]+>', num, restored_text, count=1)
        
#     restored_text = re.sub(r'<[^>]+>', '', restored_text)
#     return restored_text

# # --- HELPER: IndicBERTScore ---
# def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
#     scores = []
#     for i in range(0, len(predictions), 32):
#         batch_preds = predictions[i : i + 32]
#         batch_refs = references[i : i + 32]
        
#         with torch.no_grad():
#             input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
#             input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
#             out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
#             out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
#             out_preds = F.normalize(out_preds, p=2, dim=1)
#             out_refs = F.normalize(out_refs, p=2, dim=1)
            
#             sims = (out_preds * out_refs).sum(dim=1)
#             scores.extend(sims.cpu().tolist())

#     return sum(scores) / len(scores) if scores else 0

# # --- 1. PYTORCH ARCHITECTURE ---
# class DAE(nn.Module):
#     def __init__(self, d_in=768, d_latent=256):
#         super().__init__()
#         self.encoder = nn.Sequential(
#             nn.Linear(d_in, 512), nn.ReLU(),
#             nn.Linear(512, d_latent)
#         )
#         self.decoder = nn.Sequential(
#             nn.Linear(d_latent, 512), nn.ReLU(),
#             nn.Linear(512, d_in)
#         )

#     def forward(self, x):
#         z = self.encoder(x)
#         x_hat = self.decoder(z)
#         return x_hat, z

# class ContextualEncoder(nn.Module):
#     def __init__(self, d_model=256, nhead=4, num_layers=4):
#         super().__init__()
#         encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
#         self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
#     def forward(self, x, mask=None):
#         if mask is not None:
#             padding_mask = (mask == 0) 
#             return self.transformer_encoder(x, src_key_padding_mask=padding_mask)
#         return self.transformer_encoder(x)

# class CustomMTPipeline(nn.Module):
#     def __init__(self, dae_path, mt5_model_name="google/mt5-small"):
#         super().__init__()

#         self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#         # IndicBERT
#         self.indic = AutoModel.from_pretrained("ai4bharat/indic-bert")

#         # DAE
#         self.dae = DAE(d_in=768, d_latent=256)

#         checkpoint = torch.load(dae_path, map_location=self.device)
#         if 'model_state_dict' in checkpoint:
#             self.dae.load_state_dict(checkpoint['model_state_dict'])
#         else:
#             self.dae.load_state_dict(checkpoint)

#         for param in self.dae.parameters():
#             param.requires_grad = False

#         # Context Encoder + Projector
#         self.context_encoder = ContextualEncoder(d_model=256, nhead=4, num_layers=4)
#         self.projector = nn.Linear(256, 512)

#         # ✅ LOAD MT5 CORRECTLY
#         self.t5 = MT5ForConditionalGeneration.from_pretrained(mt5_model_name)

#         # IMPORTANT when loading custom decoder weights
#         self.t5.config.tie_word_embeddings = False

#         # Remove encoder (IndicBERT is acting as encoder)
#         del self.t5.encoder

#     def forward(self, src_ids, src_mask, tgt_ids):

#         indic_out = self.indic(src_ids, attention_mask=src_mask).last_hidden_state
#         _, latent_z = self.dae(indic_out)

#         context_out = self.context_encoder(latent_z, mask=src_mask)
#         decoder_input_embeds = self.projector(context_out)

#         encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)

#         out = self.t5(
#             encoder_outputs=encoder_outputs_wrapped,
#             labels=tgt_ids,
#             attention_mask=src_mask
#         )

#         return out.loss


# # --- 2. SETUP PIPELINE & EVAL MODELS ---
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# indic_tok = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
# t5_tok = AutoTokenizer.from_pretrained("google/mt5-small", use_fast=False)

# print("Loading Custom Pipeline Weights...")
# model = CustomMTPipeline(DAE_PATH, mt5_model_name="google/mt5-small").to(device)
# # We still load the weights so the model object initializes properly, even though we ablate the context encoder
# model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
# model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
# model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
# model.eval()

# print("Loading Global Metric Evaluators...")
# eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
# eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
# eval_model.eval()

# # --- 3. FIND UNIQUE SOURCE FILES ---
# print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
# file_pattern = f"*_to_{TARGET_LANG}.csv"
# all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

# if not all_files:
#     print("No files found! Check your directory.")
#     exit()

# # --- 4. MAIN EVALUATION LOOP (ABLATION MODE) ---
# results = []
# processed_languages = set() 

# for file_path in all_files:
#     try:
#         header_df = pd.read_csv(file_path, nrows=0)
#         cols = header_df.columns.tolist()
        
#         if TARGET_LANG not in cols:
#             continue 
            
#         src_col = [c for c in cols if c != TARGET_LANG][0]
        
#         if src_col in processed_languages:
#             continue 
        
#         processed_languages.add(src_col)
#         print(f"\n[ABLATION] Testing Pair: {src_col} -> {TARGET_LANG} (Bypassing Contextual Encoder)")
        
#         df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
#         sample_df = df.dropna().astype(str)
        
#         src_texts = sample_df[src_col].tolist()
#         ref_texts = sample_df[TARGET_LANG].tolist()
        
#         predictions = []
#         for i in range(0, len(src_texts), BATCH_SIZE):
#             batch_src = src_texts[i : i + BATCH_SIZE]
            
#             inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
#             with torch.no_grad():
#                 # 1. IndicBERT
#                 indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
#                 # 2. DAE Tokenwise Extraction
#                 _, latent_z = model.dae(indic_out)
                
#                 # =========================================================
#                 # 🚨 ABLATION OVERRIDE: BYPASS THE CONTEXTUAL ENCODER 🚨
#                 # Instead of passing latent_z to context_encoder, we send it 
#                 # directly to the projector to test translation without sentence-level context.
#                 # =========================================================
#                 decoder_input_embeds = model.projector(latent_z)
                
#                 encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
#                 # 3. Target Generation
#                 gen_ids = model.t5.generate(
#                     encoder_outputs=encoder_outputs_wrapped,
#                     attention_mask=inputs.attention_mask,
#                     max_length=64,
#                     num_beams=4,
#                     do_sample=False
#                 )
                
#             batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            
#             # Post-processing to fix placeholder tags
#             for j in range(len(batch_preds)):
#                 batch_preds[j] = restore_masked_numbers(batch_src[j], batch_preds[j])
                
#             predictions.extend(batch_preds)
            
#         # 🕵️ DIAGNOSTIC BLOCK
#         print("\n" + "="*70)
#         print(f"🕵️ ABLATION SAMPLES FOR ({src_col} -> {TARGET_LANG}) 🕵️")
#         print("="*70)

#         sample_indices = random.sample(range(len(predictions)), min(5, len(predictions)))

#         for idx, rand_i in enumerate(sample_indices):
#             print(f"\n--- Sample {idx+1} ---")
#             print(f"Source ({src_col}):      {src_texts[rand_i]}")
#             print(f"Reference ({TARGET_LANG}):   {ref_texts[rand_i]}")
#             print(f"Prediction ({TARGET_LANG}):  {predictions[rand_i]}")

#         print("\n" + "="*70)
            
#         # Calculate Metrics
#         bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
#         chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
#         bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
#         print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
#         results.append({
#             "Source Language": src_col,
#             "BLEU": bleu.score,
#             "ChrF++": chrf.score,
#             "IndicBERT": bert
#         })
        
#     except Exception as e:
#         print(f"Error processing {os.path.basename(file_path)}: {e}")

# # --- 5. FINAL REPORT ---
# if results:
#     results_df = pd.DataFrame(results)
    
#     print("\n" + "="*70)
#     print(f"FINAL AGGREGATE RESULTS FOR ABLATION (No Contextual Encoder): {TARGET_LANG}")
#     print("="*70)
#     print(results_df.round(2).to_string(index=False))
#     print("-" * 70)
    
#     avg_bleu = results_df["BLEU"].mean()
#     avg_chrf = results_df["ChrF++"].mean()
#     avg_bert = results_df["IndicBERT"].mean()
    
#     print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
#     print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
#     print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
#     print("="*70)
    
#     # Save with a specific ablation filename
#     results_df.to_csv(f"ablation_no_context_results_{TARGET_LANG}.csv", index=False)
#     print(f"✓ Ablation results saved to ablation_no_context_results_{TARGET_LANG}.csv")
# else:
#     print("No results generated.")

# Malyalam

In [4]:
# import torch
# import torch.nn as nn
# import pandas as pd
# import sacrebleu
# import glob
# import os
# import random
# import re
# import numpy as np
# from tqdm import tqdm
# from transformers import AutoModel, AutoTokenizer, MT5ForConditionalGeneration
# from transformers.modeling_outputs import BaseModelOutput
# import torch.nn.functional as F
# from huggingface_hub import login
# from kaggle_secrets import UserSecretsClient

# # Fetch the secret token
# user_secrets = UserSecretsClient()
# hf_token = user_secrets.get_secret("HF_TOKEN")

# # Login automatically
# login(token=hf_token, add_to_git_credential=False)

# # --- CONFIGURATION ---
# TARGET_LANG = "ml"  # The Decoder Language we are testing
# DECODER_WEIGHTS = "/kaggle/input/models/akashbardia/decoder-mal-mlym/pytorch/default/1/decoder_mal_Mlym.pt"
# DAE_PATH = '/kaggle/input/models/vedantm23/dae-model-epoch-7/pytorch/finalmodel/1/dae_model_final.pt'

# # Shared Paths (Universal Encoder)
# SHARED_ENC_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_context_encoder.pt"
# SHARED_PROJ_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_projector.pt"

# BATCH_SIZE = 16

# # --- HELPER: RESTORE MASKED NUMBERS ---
# def restore_masked_numbers(source_text, predicted_text):
#     """
#     Extracts numbers/dates from the source text and replaces placeholder tags 
#     (like <আইডি১> or <ID1>) in the predicted text.
#     """
#     source_numbers = re.findall(r'[\d\.]+\d', source_text)
#     restored_text = predicted_text
    
#     for num in source_numbers:
#         restored_text = re.sub(r'<[^>]+>', num, restored_text, count=1)
        
#     restored_text = re.sub(r'<[^>]+>', '', restored_text)
#     return restored_text

# # --- HELPER: IndicBERTScore ---
# def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
#     scores = []
#     for i in range(0, len(predictions), 32):
#         batch_preds = predictions[i : i + 32]
#         batch_refs = references[i : i + 32]
        
#         with torch.no_grad():
#             input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
#             input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
#             out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
#             out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
#             out_preds = F.normalize(out_preds, p=2, dim=1)
#             out_refs = F.normalize(out_refs, p=2, dim=1)
            
#             sims = (out_preds * out_refs).sum(dim=1)
#             scores.extend(sims.cpu().tolist())

#     return sum(scores) / len(scores) if scores else 0

# # --- 1. PYTORCH ARCHITECTURE ---
# class DAE(nn.Module):
#     def __init__(self, d_in=768, d_latent=256):
#         super().__init__()
#         self.encoder = nn.Sequential(
#             nn.Linear(d_in, 512), nn.ReLU(),
#             nn.Linear(512, d_latent)
#         )
#         self.decoder = nn.Sequential(
#             nn.Linear(d_latent, 512), nn.ReLU(),
#             nn.Linear(512, d_in)
#         )

#     def forward(self, x):
#         z = self.encoder(x)
#         x_hat = self.decoder(z)
#         return x_hat, z

# class ContextualEncoder(nn.Module):
#     def __init__(self, d_model=256, nhead=4, num_layers=4):
#         super().__init__()
#         encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
#         self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
#     def forward(self, x, mask=None):
#         if mask is not None:
#             padding_mask = (mask == 0) 
#             return self.transformer_encoder(x, src_key_padding_mask=padding_mask)
#         return self.transformer_encoder(x)

# class CustomMTPipeline(nn.Module):
#     def __init__(self, dae_path, mt5_model_name="google/mt5-small"):
#         super().__init__()

#         self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#         # IndicBERT
#         self.indic = AutoModel.from_pretrained("ai4bharat/indic-bert")

#         # DAE
#         self.dae = DAE(d_in=768, d_latent=256)

#         checkpoint = torch.load(dae_path, map_location=self.device)
#         if 'model_state_dict' in checkpoint:
#             self.dae.load_state_dict(checkpoint['model_state_dict'])
#         else:
#             self.dae.load_state_dict(checkpoint)

#         for param in self.dae.parameters():
#             param.requires_grad = False

#         # Context Encoder + Projector
#         self.context_encoder = ContextualEncoder(d_model=256, nhead=4, num_layers=4)
#         self.projector = nn.Linear(256, 512)

#         # ✅ LOAD MT5 CORRECTLY
#         self.t5 = MT5ForConditionalGeneration.from_pretrained(mt5_model_name)

#         # IMPORTANT when loading custom decoder weights
#         self.t5.config.tie_word_embeddings = False

#         # Remove encoder (IndicBERT is acting as encoder)
#         del self.t5.encoder

#     def forward(self, src_ids, src_mask, tgt_ids):

#         indic_out = self.indic(src_ids, attention_mask=src_mask).last_hidden_state
#         _, latent_z = self.dae(indic_out)

#         context_out = self.context_encoder(latent_z, mask=src_mask)
#         decoder_input_embeds = self.projector(context_out)

#         encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)

#         out = self.t5(
#             encoder_outputs=encoder_outputs_wrapped,
#             labels=tgt_ids,
#             attention_mask=src_mask
#         )

#         return out.loss


# # --- 2. SETUP PIPELINE & EVAL MODELS ---
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# indic_tok = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
# t5_tok = AutoTokenizer.from_pretrained("google/mt5-small", use_fast=False)

# print("Loading Custom Pipeline Weights...")
# model = CustomMTPipeline(DAE_PATH, mt5_model_name="google/mt5-small").to(device)
# # We still load the weights so the model object initializes properly, even though we ablate the context encoder
# model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
# model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
# model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
# model.eval()

# print("Loading Global Metric Evaluators...")
# eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
# eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
# eval_model.eval()

# # --- 3. FIND UNIQUE SOURCE FILES ---
# print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
# file_pattern = f"*_to_{TARGET_LANG}.csv"
# all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

# if not all_files:
#     print("No files found! Check your directory.")
#     exit()

# # --- 4. MAIN EVALUATION LOOP (ABLATION MODE) ---
# results = []
# processed_languages = set() 

# for file_path in all_files:
#     try:
#         header_df = pd.read_csv(file_path, nrows=0)
#         cols = header_df.columns.tolist()
        
#         if TARGET_LANG not in cols:
#             continue 
            
#         src_col = [c for c in cols if c != TARGET_LANG][0]
        
#         if src_col in processed_languages:
#             continue 
        
#         processed_languages.add(src_col)
#         print(f"\n[ABLATION] Testing Pair: {src_col} -> {TARGET_LANG} (Bypassing Contextual Encoder)")
        
#         df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
#         sample_df = df.dropna().astype(str)
        
#         src_texts = sample_df[src_col].tolist()
#         ref_texts = sample_df[TARGET_LANG].tolist()
        
#         predictions = []
#         for i in range(0, len(src_texts), BATCH_SIZE):
#             batch_src = src_texts[i : i + BATCH_SIZE]
            
#             inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
#             with torch.no_grad():
#                 # 1. IndicBERT
#                 indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
#                 # 2. DAE Tokenwise Extraction
#                 _, latent_z = model.dae(indic_out)
                
#                 # =========================================================
#                 # 🚨 ABLATION OVERRIDE: BYPASS THE CONTEXTUAL ENCODER 🚨
#                 # Instead of passing latent_z to context_encoder, we send it 
#                 # directly to the projector to test translation without sentence-level context.
#                 # =========================================================
#                 decoder_input_embeds = model.projector(latent_z)
                
#                 encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
#                 # 3. Target Generation
#                 gen_ids = model.t5.generate(
#                     encoder_outputs=encoder_outputs_wrapped,
#                     attention_mask=inputs.attention_mask,
#                     max_length=64,
#                     num_beams=4,
#                     do_sample=False
#                 )
                
#             batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            
#             # Post-processing to fix placeholder tags
#             for j in range(len(batch_preds)):
#                 batch_preds[j] = restore_masked_numbers(batch_src[j], batch_preds[j])
                
#             predictions.extend(batch_preds)
            
#         # 🕵️ DIAGNOSTIC BLOCK
#         print("\n" + "="*70)
#         print(f"🕵️ ABLATION SAMPLES FOR ({src_col} -> {TARGET_LANG}) 🕵️")
#         print("="*70)

#         sample_indices = random.sample(range(len(predictions)), min(5, len(predictions)))

#         for idx, rand_i in enumerate(sample_indices):
#             print(f"\n--- Sample {idx+1} ---")
#             print(f"Source ({src_col}):      {src_texts[rand_i]}")
#             print(f"Reference ({TARGET_LANG}):   {ref_texts[rand_i]}")
#             print(f"Prediction ({TARGET_LANG}):  {predictions[rand_i]}")

#         print("\n" + "="*70)
            
#         # Calculate Metrics
#         bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
#         chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
#         bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
#         print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
#         results.append({
#             "Source Language": src_col,
#             "BLEU": bleu.score,
#             "ChrF++": chrf.score,
#             "IndicBERT": bert
#         })
        
#     except Exception as e:
#         print(f"Error processing {os.path.basename(file_path)}: {e}")

# # --- 5. FINAL REPORT ---
# if results:
#     results_df = pd.DataFrame(results)
    
#     print("\n" + "="*70)
#     print(f"FINAL AGGREGATE RESULTS FOR ABLATION (No Contextual Encoder): {TARGET_LANG}")
#     print("="*70)
#     print(results_df.round(2).to_string(index=False))
#     print("-" * 70)
    
#     avg_bleu = results_df["BLEU"].mean()
#     avg_chrf = results_df["ChrF++"].mean()
#     avg_bert = results_df["IndicBERT"].mean()
    
#     print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
#     print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
#     print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
#     print("="*70)
    
#     # Save with a specific ablation filename
#     results_df.to_csv(f"ablation_no_context_results_{TARGET_LANG}.csv", index=False)
#     print(f"✓ Ablation results saved to ablation_no_context_results_{TARGET_LANG}.csv")
# else:
#     print("No results generated.")

# Marathi

In [5]:
import torch
import torch.nn as nn
import pandas as pd
import sacrebleu
import glob
import os
import random
import re
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, MT5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "mr"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/models/drneelimasingh/decoder-mar/pytorch/default/1/decoder_mar_Deva.pt"
DAE_PATH = '/kaggle/input/models/vedantm23/dae-model-epoch-7/pytorch/finalmodel/1/dae_model_final.pt'

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_projector.pt"

BATCH_SIZE = 16

# --- HELPER: RESTORE MASKED NUMBERS ---
def restore_masked_numbers(source_text, predicted_text):
    """
    Extracts numbers/dates from the source text and replaces placeholder tags 
    (like <আইডি১> or <ID1>) in the predicted text.
    """
    source_numbers = re.findall(r'[\d\.]+\d', source_text)
    restored_text = predicted_text
    
    for num in source_numbers:
        restored_text = re.sub(r'<[^>]+>', num, restored_text, count=1)
        
    restored_text = re.sub(r'<[^>]+>', '', restored_text)
    return restored_text

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. PYTORCH ARCHITECTURE ---
class DAE(nn.Module):
    def __init__(self, d_in=768, d_latent=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, 512), nn.ReLU(),
            nn.Linear(512, d_latent)
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, 512), nn.ReLU(),
            nn.Linear(512, d_in)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

class ContextualEncoder(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_layers=4):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
    def forward(self, x, mask=None):
        if mask is not None:
            padding_mask = (mask == 0) 
            return self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        return self.transformer_encoder(x)

class CustomMTPipeline(nn.Module):
    def __init__(self, dae_path, mt5_model_name="google/mt5-small"):
        super().__init__()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # IndicBERT
        self.indic = AutoModel.from_pretrained("ai4bharat/indic-bert")

        # DAE
        self.dae = DAE(d_in=768, d_latent=256)

        checkpoint = torch.load(dae_path, map_location=self.device)
        if 'model_state_dict' in checkpoint:
            self.dae.load_state_dict(checkpoint['model_state_dict'])
        else:
            self.dae.load_state_dict(checkpoint)

        for param in self.dae.parameters():
            param.requires_grad = False

        # Context Encoder + Projector
        self.context_encoder = ContextualEncoder(d_model=256, nhead=4, num_layers=4)
        self.projector = nn.Linear(256, 512)

        # ✅ LOAD MT5 CORRECTLY
        self.t5 = MT5ForConditionalGeneration.from_pretrained(mt5_model_name)

        # IMPORTANT when loading custom decoder weights
        self.t5.config.tie_word_embeddings = False

        # Remove encoder (IndicBERT is acting as encoder)
        del self.t5.encoder

    def forward(self, src_ids, src_mask, tgt_ids):

        indic_out = self.indic(src_ids, attention_mask=src_mask).last_hidden_state
        _, latent_z = self.dae(indic_out)

        context_out = self.context_encoder(latent_z, mask=src_mask)
        decoder_input_embeds = self.projector(context_out)

        encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)

        out = self.t5(
            encoder_outputs=encoder_outputs_wrapped,
            labels=tgt_ids,
            attention_mask=src_mask
        )

        return out.loss


# --- 2. SETUP PIPELINE & EVAL MODELS ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

indic_tok = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
t5_tok = AutoTokenizer.from_pretrained("google/mt5-small", use_fast=False)

print("Loading Custom Pipeline Weights...")
model = CustomMTPipeline(DAE_PATH, mt5_model_name="google/mt5-small").to(device)
# We still load the weights so the model object initializes properly, even though we ablate the context encoder
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

print("Loading Global Metric Evaluators...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()

# --- 3. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 4. MAIN EVALUATION LOOP (ABLATION MODE) ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        print(f"\n[ABLATION] Testing Pair: {src_col} -> {TARGET_LANG} (Bypassing Contextual Encoder)")
        
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                # 1. IndicBERT
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                # 2. DAE Tokenwise Extraction
                _, latent_z = model.dae(indic_out)
                
                # =========================================================
                # 🚨 ABLATION OVERRIDE: BYPASS THE CONTEXTUAL ENCODER 🚨
                # Instead of passing latent_z to context_encoder, we send it 
                # directly to the projector to test translation without sentence-level context.
                # =========================================================
                decoder_input_embeds = model.projector(latent_z)
                
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                # 3. Target Generation
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped,
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    do_sample=False
                )
                
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            
            # Post-processing to fix placeholder tags
            for j in range(len(batch_preds)):
                batch_preds[j] = restore_masked_numbers(batch_src[j], batch_preds[j])
                
            predictions.extend(batch_preds)
            
        # 🕵️ DIAGNOSTIC BLOCK
        print("\n" + "="*70)
        print(f"🕵️ ABLATION SAMPLES FOR ({src_col} -> {TARGET_LANG}) 🕵️")
        print("="*70)

        sample_indices = random.sample(range(len(predictions)), min(5, len(predictions)))

        for idx, rand_i in enumerate(sample_indices):
            print(f"\n--- Sample {idx+1} ---")
            print(f"Source ({src_col}):      {src_texts[rand_i]}")
            print(f"Reference ({TARGET_LANG}):   {ref_texts[rand_i]}")
            print(f"Prediction ({TARGET_LANG}):  {predictions[rand_i]}")

        print("\n" + "="*70)
            
        # Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 5. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR ABLATION (No Contextual Encoder): {TARGET_LANG}")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    # Save with a specific ablation filename
    results_df.to_csv(f"ablation_no_context_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Ablation results saved to ablation_no_context_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

Using device: cuda


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading Global Metric Evaluators...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Searching for pairs ending in ..._to_mr.csv

[ABLATION] Testing Pair: bn -> mr (Bypassing Contextual Encoder)

🕵️ ABLATION SAMPLES FOR (bn -> mr) 🕵️

--- Sample 1 ---
Source (bn):      ২০২১-এর মহিলা রাগবি বিশ্বকাপের নবম সংস্করণটি পূর্বপরিকল্পিত ১৮ই সেপ্টেম্বর থেকে ১৮ই অক্টোবরের সময়সূচি থেকে পিছিয়ে ৮ই অক্টোবর থেকে ১২ই নভেম্বর ২০২২-এ নিউজিল্যাণ্ডে অনুষ্ঠিত হয়।
Reference (mr):   २०२१ची रग्बी विश्वचषक स्पर्धा ही महिलांच्या रग्बी विश्वचषक स्पर्धेची नववी आवृत्ती ८ ऑक्टोबर आणि १२ नव्हेंबर २०२२ दरम्यान न्यूझीलंडमध्ये आयोजित केली गेली, जी मुळात १८ सप्टेंबर ते १८ ऑक्टोबर २०२१ दरम्यान नियोजित असलेल्या वेळापत्रकापासून पुढे ढकलली गेली.
Prediction (mr):  सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी सकाळी

--- Sample 2 ---
Source (bn):      সমাপ্তির কাছাকাছি একটি ছোট অংশ যুদ্ধবিগ্রহের বৃহত্তর বিষয়গুলিকে উদ্দেশ করে এবং রণহস্তী এবং সৈনিকের বিভিন্ন ব্যবহার ব্যাখ্যা করে।
Reference (mr):   शेवटचा एक छोटा उतारा, युद्धकलेसंबंधित महत्

# Gujrati

In [6]:
import torch
import torch.nn as nn
import pandas as pd
import sacrebleu
import glob
import os
import random
import re
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, MT5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "gu"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/models/drneelimasingh/decoder-gur-and-orya/pytorch/default/1/decoder_guj_Gujr.pt"
DAE_PATH = '/kaggle/input/models/vedantm23/dae-model-epoch-7/pytorch/finalmodel/1/dae_model_final.pt'

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_projector.pt"

BATCH_SIZE = 16

# --- HELPER: RESTORE MASKED NUMBERS ---
def restore_masked_numbers(source_text, predicted_text):
    """
    Extracts numbers/dates from the source text and replaces placeholder tags 
    (like <আইডি১> or <ID1>) in the predicted text.
    """
    source_numbers = re.findall(r'[\d\.]+\d', source_text)
    restored_text = predicted_text
    
    for num in source_numbers:
        restored_text = re.sub(r'<[^>]+>', num, restored_text, count=1)
        
    restored_text = re.sub(r'<[^>]+>', '', restored_text)
    return restored_text

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. PYTORCH ARCHITECTURE ---
class DAE(nn.Module):
    def __init__(self, d_in=768, d_latent=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, 512), nn.ReLU(),
            nn.Linear(512, d_latent)
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, 512), nn.ReLU(),
            nn.Linear(512, d_in)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

class ContextualEncoder(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_layers=4):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
    def forward(self, x, mask=None):
        if mask is not None:
            padding_mask = (mask == 0) 
            return self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        return self.transformer_encoder(x)

class CustomMTPipeline(nn.Module):
    def __init__(self, dae_path, mt5_model_name="google/mt5-small"):
        super().__init__()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # IndicBERT
        self.indic = AutoModel.from_pretrained("ai4bharat/indic-bert")

        # DAE
        self.dae = DAE(d_in=768, d_latent=256)

        checkpoint = torch.load(dae_path, map_location=self.device)
        if 'model_state_dict' in checkpoint:
            self.dae.load_state_dict(checkpoint['model_state_dict'])
        else:
            self.dae.load_state_dict(checkpoint)

        for param in self.dae.parameters():
            param.requires_grad = False

        # Context Encoder + Projector
        self.context_encoder = ContextualEncoder(d_model=256, nhead=4, num_layers=4)
        self.projector = nn.Linear(256, 512)

        # ✅ LOAD MT5 CORRECTLY
        self.t5 = MT5ForConditionalGeneration.from_pretrained(mt5_model_name)

        # IMPORTANT when loading custom decoder weights
        self.t5.config.tie_word_embeddings = False

        # Remove encoder (IndicBERT is acting as encoder)
        del self.t5.encoder

    def forward(self, src_ids, src_mask, tgt_ids):

        indic_out = self.indic(src_ids, attention_mask=src_mask).last_hidden_state
        _, latent_z = self.dae(indic_out)

        context_out = self.context_encoder(latent_z, mask=src_mask)
        decoder_input_embeds = self.projector(context_out)

        encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)

        out = self.t5(
            encoder_outputs=encoder_outputs_wrapped,
            labels=tgt_ids,
            attention_mask=src_mask
        )

        return out.loss


# --- 2. SETUP PIPELINE & EVAL MODELS ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

indic_tok = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
t5_tok = AutoTokenizer.from_pretrained("google/mt5-small", use_fast=False)

print("Loading Custom Pipeline Weights...")
model = CustomMTPipeline(DAE_PATH, mt5_model_name="google/mt5-small").to(device)
# We still load the weights so the model object initializes properly, even though we ablate the context encoder
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

print("Loading Global Metric Evaluators...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()

# --- 3. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 4. MAIN EVALUATION LOOP (ABLATION MODE) ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        print(f"\n[ABLATION] Testing Pair: {src_col} -> {TARGET_LANG} (Bypassing Contextual Encoder)")
        
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                # 1. IndicBERT
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                # 2. DAE Tokenwise Extraction
                _, latent_z = model.dae(indic_out)
                
                # =========================================================
                # 🚨 ABLATION OVERRIDE: BYPASS THE CONTEXTUAL ENCODER 🚨
                # Instead of passing latent_z to context_encoder, we send it 
                # directly to the projector to test translation without sentence-level context.
                # =========================================================
                decoder_input_embeds = model.projector(latent_z)
                
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                # 3. Target Generation
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped,
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    do_sample=False
                )
                
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            
            # Post-processing to fix placeholder tags
            for j in range(len(batch_preds)):
                batch_preds[j] = restore_masked_numbers(batch_src[j], batch_preds[j])
                
            predictions.extend(batch_preds)
            
        # 🕵️ DIAGNOSTIC BLOCK
        print("\n" + "="*70)
        print(f"🕵️ ABLATION SAMPLES FOR ({src_col} -> {TARGET_LANG}) 🕵️")
        print("="*70)

        sample_indices = random.sample(range(len(predictions)), min(5, len(predictions)))

        for idx, rand_i in enumerate(sample_indices):
            print(f"\n--- Sample {idx+1} ---")
            print(f"Source ({src_col}):      {src_texts[rand_i]}")
            print(f"Reference ({TARGET_LANG}):   {ref_texts[rand_i]}")
            print(f"Prediction ({TARGET_LANG}):  {predictions[rand_i]}")

        print("\n" + "="*70)
            
        # Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 5. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR ABLATION (No Contextual Encoder): {TARGET_LANG}")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    # Save with a specific ablation filename
    results_df.to_csv(f"ablation_no_context_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Ablation results saved to ablation_no_context_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

Using device: cuda
Loading Custom Pipeline Weights...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading Global Metric Evaluators...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Searching for pairs ending in ..._to_gu.csv

[ABLATION] Testing Pair: mr -> gu (Bypassing Contextual Encoder)

🕵️ ABLATION SAMPLES FOR (mr -> gu) 🕵️

--- Sample 1 ---
Source (mr):      व्यापारी समतोल म्हणजे, वस्तू आणि सेवांच्या निर्यातीचे मूल्य आणि वस्तू आणि सेवांच्या आयातीचे मूल्य यांच्यातील फरक.
Reference (gu):   વ્યાપાર સંતુલન એ માલસામાન અને સેવાઓના નિકાસ મુલ્ય અને માલસામાન અને સેવાઓના આયાત મુલ્ય વચ્ચેનો તફાવત છે.
Prediction (gu):  ટૂંક સમયમાં, ટૂંક સમયમાં, ટૂંક સમયમાં, ટૂંક સમયમાં, ટૂંક સમયમાં, ટૂંક સમયમાં, ટૂંક સમયમાં, ટૂંક સમયમાં

--- Sample 2 ---
Source (mr):      एप्रिल २०२०च्या सुरूवातीला, संचालक डॉ. जॉन नकेङ्गासोन यांनी प्रोफेसर जीन-पॉल मीरा आणि कॅमिली लोख्ट या दोन फ्रेंच शास्त्रज्ञांच्या टिप्पण्यांचे खंडण केले, ज्यात त्यांनी, आफ्रिकेत क्षयरोगाच्या एका संभाव्य लसीची कोरोना विषाणूसाठी चाचणी करणे "आक्षेपार्ह आणि वर्णवादी" असल्याचे म्हटले होते.
Reference (gu):   એપ્રિલ, ૨૦૨૦ના આરંભે નિર્દેશક ડો. જ્હોન કેન્ગાસોંગએ બે ફ્રેંચ વૈજ્ઞાનિકો, પ્રોફેસર જીન-પોલ મીરા અને કેમિલ લોચની એવી ટી

# Tamil

In [7]:
import torch
import torch.nn as nn
import pandas as pd
import sacrebleu
import glob
import os
import random
import re
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, MT5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "ta"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/models/waibhavjha/decoder-tam-taml/pytorch/default/1/decoder_tam_Taml.pt"
DAE_PATH = '/kaggle/input/models/vedantm23/dae-model-epoch-7/pytorch/finalmodel/1/dae_model_final.pt'

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_projector.pt"

BATCH_SIZE = 16

# --- HELPER: RESTORE MASKED NUMBERS ---
def restore_masked_numbers(source_text, predicted_text):
    """
    Extracts numbers/dates from the source text and replaces placeholder tags 
    (like <আইডি১> or <ID1>) in the predicted text.
    """
    source_numbers = re.findall(r'[\d\.]+\d', source_text)
    restored_text = predicted_text
    
    for num in source_numbers:
        restored_text = re.sub(r'<[^>]+>', num, restored_text, count=1)
        
    restored_text = re.sub(r'<[^>]+>', '', restored_text)
    return restored_text

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. PYTORCH ARCHITECTURE ---
class DAE(nn.Module):
    def __init__(self, d_in=768, d_latent=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, 512), nn.ReLU(),
            nn.Linear(512, d_latent)
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, 512), nn.ReLU(),
            nn.Linear(512, d_in)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

class ContextualEncoder(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_layers=4):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
    def forward(self, x, mask=None):
        if mask is not None:
            padding_mask = (mask == 0) 
            return self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        return self.transformer_encoder(x)

class CustomMTPipeline(nn.Module):
    def __init__(self, dae_path, mt5_model_name="google/mt5-small"):
        super().__init__()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # IndicBERT
        self.indic = AutoModel.from_pretrained("ai4bharat/indic-bert")

        # DAE
        self.dae = DAE(d_in=768, d_latent=256)

        checkpoint = torch.load(dae_path, map_location=self.device)
        if 'model_state_dict' in checkpoint:
            self.dae.load_state_dict(checkpoint['model_state_dict'])
        else:
            self.dae.load_state_dict(checkpoint)

        for param in self.dae.parameters():
            param.requires_grad = False

        # Context Encoder + Projector
        self.context_encoder = ContextualEncoder(d_model=256, nhead=4, num_layers=4)
        self.projector = nn.Linear(256, 512)

        # ✅ LOAD MT5 CORRECTLY
        self.t5 = MT5ForConditionalGeneration.from_pretrained(mt5_model_name)

        # IMPORTANT when loading custom decoder weights
        self.t5.config.tie_word_embeddings = False

        # Remove encoder (IndicBERT is acting as encoder)
        del self.t5.encoder

    def forward(self, src_ids, src_mask, tgt_ids):

        indic_out = self.indic(src_ids, attention_mask=src_mask).last_hidden_state
        _, latent_z = self.dae(indic_out)

        context_out = self.context_encoder(latent_z, mask=src_mask)
        decoder_input_embeds = self.projector(context_out)

        encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)

        out = self.t5(
            encoder_outputs=encoder_outputs_wrapped,
            labels=tgt_ids,
            attention_mask=src_mask
        )

        return out.loss


# --- 2. SETUP PIPELINE & EVAL MODELS ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

indic_tok = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
t5_tok = AutoTokenizer.from_pretrained("google/mt5-small", use_fast=False)

print("Loading Custom Pipeline Weights...")
model = CustomMTPipeline(DAE_PATH, mt5_model_name="google/mt5-small").to(device)
# We still load the weights so the model object initializes properly, even though we ablate the context encoder
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

print("Loading Global Metric Evaluators...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()

# --- 3. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 4. MAIN EVALUATION LOOP (ABLATION MODE) ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        print(f"\n[ABLATION] Testing Pair: {src_col} -> {TARGET_LANG} (Bypassing Contextual Encoder)")
        
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                # 1. IndicBERT
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                # 2. DAE Tokenwise Extraction
                _, latent_z = model.dae(indic_out)
                
                # =========================================================
                # 🚨 ABLATION OVERRIDE: BYPASS THE CONTEXTUAL ENCODER 🚨
                # Instead of passing latent_z to context_encoder, we send it 
                # directly to the projector to test translation without sentence-level context.
                # =========================================================
                decoder_input_embeds = model.projector(latent_z)
                
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                # 3. Target Generation
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped,
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    do_sample=False
                )
                
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            
            # Post-processing to fix placeholder tags
            for j in range(len(batch_preds)):
                batch_preds[j] = restore_masked_numbers(batch_src[j], batch_preds[j])
                
            predictions.extend(batch_preds)
            
        # 🕵️ DIAGNOSTIC BLOCK
        print("\n" + "="*70)
        print(f"🕵️ ABLATION SAMPLES FOR ({src_col} -> {TARGET_LANG}) 🕵️")
        print("="*70)

        sample_indices = random.sample(range(len(predictions)), min(5, len(predictions)))

        for idx, rand_i in enumerate(sample_indices):
            print(f"\n--- Sample {idx+1} ---")
            print(f"Source ({src_col}):      {src_texts[rand_i]}")
            print(f"Reference ({TARGET_LANG}):   {ref_texts[rand_i]}")
            print(f"Prediction ({TARGET_LANG}):  {predictions[rand_i]}")

        print("\n" + "="*70)
            
        # Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 5. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR ABLATION (No Contextual Encoder): {TARGET_LANG}")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    # Save with a specific ablation filename
    results_df.to_csv(f"ablation_no_context_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Ablation results saved to ablation_no_context_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

Using device: cuda
Loading Custom Pipeline Weights...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading Global Metric Evaluators...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Searching for pairs ending in ..._to_ta.csv

[ABLATION] Testing Pair: as -> ta (Bypassing Contextual Encoder)

🕵️ ABLATION SAMPLES FOR (as -> ta) 🕵️

--- Sample 1 ---
Source (as):      ইয়াৰ প্ৰৱালৰ টিলাবোৰ সাধাৰণতে কম বহল আৰু ইয়াৰ পূব প্ৰান্তবোৰৰ বাহিৰে ভিলুৰ কোনো চিন দেখা নাযায় যিবোৰ মালদ্বীপৰ স্তৰ অনুসৰি যঠেষ্ট বহল দ্বীপেৰে আবৃত৷
Reference (ta):   அதன் நீரடித் திட்டுகள் பொதுவாக வெகு குறைந்த அகலமே கொண்டவை மற்றும் விலுவின் அறிகுறியே இல்லாதவை, அதன் கிழக்கு விளிம்புப் பகுதிகள் மட்டுமே இதற்கு விதிவிலக்கு, அங்கு மாலத்தீவு அளவுகோலின் படி மிகவும் பெரிய தீவுகள் சூழ்ந்துள்ளன.
Prediction (ta):  இளைய சகோதரர் ü0B3C Âu0B3C Âu0B3C Âu0B3C Âu0B3C Âu0B3C Âu0B3C Âu0B3C Âu0

--- Sample 2 ---
Source (as):      ন্যায় আৰু দাসত্বৰ পৰা স্বাধীনতাৰ প্ৰতি মানুহক জাগ্রত কৰাৰ উদ্দেশ্যেৰে লিখাটো লেখকসকলৰ বাবে এক সামাজিক দায়বদ্ধতাত পৰিণত হৈছিল।
Reference (ta):   நீதி, அடிமைத்தனத்திலிருந்து விடுதலை ஆகிய சிந்தனைகள் பற்றி மக்களுக்கு விழிப்புணர்வு ஏற்படுத்தும் நோக்கில் எழுதுவது எழுத்தாளர்களின் சமூகப் பொறுப்பாக மாறியத

# Orya

In [8]:
import torch
import torch.nn as nn
import pandas as pd
import sacrebleu
import glob
import os
import random
import re
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, MT5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "or"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/models/drneelimasingh/decoder-gur-and-orya/pytorch/default/1/decoder_ory_Orya.pt"
DAE_PATH = '/kaggle/input/models/vedantm23/dae-model-epoch-7/pytorch/finalmodel/1/dae_model_final.pt'

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/datasets/drneelimasingh/encoder-output/shared_projector.pt"

BATCH_SIZE = 16

# --- HELPER: RESTORE MASKED NUMBERS ---
def restore_masked_numbers(source_text, predicted_text):
    """
    Extracts numbers/dates from the source text and replaces placeholder tags 
    (like <আইডি১> or <ID1>) in the predicted text.
    """
    source_numbers = re.findall(r'[\d\.]+\d', source_text)
    restored_text = predicted_text
    
    for num in source_numbers:
        restored_text = re.sub(r'<[^>]+>', num, restored_text, count=1)
        
    restored_text = re.sub(r'<[^>]+>', '', restored_text)
    return restored_text

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. PYTORCH ARCHITECTURE ---
class DAE(nn.Module):
    def __init__(self, d_in=768, d_latent=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, 512), nn.ReLU(),
            nn.Linear(512, d_latent)
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, 512), nn.ReLU(),
            nn.Linear(512, d_in)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

class ContextualEncoder(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_layers=4):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
    def forward(self, x, mask=None):
        if mask is not None:
            padding_mask = (mask == 0) 
            return self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        return self.transformer_encoder(x)

class CustomMTPipeline(nn.Module):
    def __init__(self, dae_path, mt5_model_name="google/mt5-small"):
        super().__init__()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # IndicBERT
        self.indic = AutoModel.from_pretrained("ai4bharat/indic-bert")

        # DAE
        self.dae = DAE(d_in=768, d_latent=256)

        checkpoint = torch.load(dae_path, map_location=self.device)
        if 'model_state_dict' in checkpoint:
            self.dae.load_state_dict(checkpoint['model_state_dict'])
        else:
            self.dae.load_state_dict(checkpoint)

        for param in self.dae.parameters():
            param.requires_grad = False

        # Context Encoder + Projector
        self.context_encoder = ContextualEncoder(d_model=256, nhead=4, num_layers=4)
        self.projector = nn.Linear(256, 512)

        # ✅ LOAD MT5 CORRECTLY
        self.t5 = MT5ForConditionalGeneration.from_pretrained(mt5_model_name)

        # IMPORTANT when loading custom decoder weights
        self.t5.config.tie_word_embeddings = False

        # Remove encoder (IndicBERT is acting as encoder)
        del self.t5.encoder

    def forward(self, src_ids, src_mask, tgt_ids):

        indic_out = self.indic(src_ids, attention_mask=src_mask).last_hidden_state
        _, latent_z = self.dae(indic_out)

        context_out = self.context_encoder(latent_z, mask=src_mask)
        decoder_input_embeds = self.projector(context_out)

        encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)

        out = self.t5(
            encoder_outputs=encoder_outputs_wrapped,
            labels=tgt_ids,
            attention_mask=src_mask
        )

        return out.loss


# --- 2. SETUP PIPELINE & EVAL MODELS ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

indic_tok = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
t5_tok = AutoTokenizer.from_pretrained("google/mt5-small", use_fast=False)

print("Loading Custom Pipeline Weights...")
model = CustomMTPipeline(DAE_PATH, mt5_model_name="google/mt5-small").to(device)
# We still load the weights so the model object initializes properly, even though we ablate the context encoder
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

print("Loading Global Metric Evaluators...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()

# --- 3. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 4. MAIN EVALUATION LOOP (ABLATION MODE) ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        print(f"\n[ABLATION] Testing Pair: {src_col} -> {TARGET_LANG} (Bypassing Contextual Encoder)")
        
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                # 1. IndicBERT
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                # 2. DAE Tokenwise Extraction
                _, latent_z = model.dae(indic_out)
                
                # =========================================================
                # 🚨 ABLATION OVERRIDE: BYPASS THE CONTEXTUAL ENCODER 🚨
                # Instead of passing latent_z to context_encoder, we send it 
                # directly to the projector to test translation without sentence-level context.
                # =========================================================
                decoder_input_embeds = model.projector(latent_z)
                
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                # 3. Target Generation
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped,
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    do_sample=False
                )
                
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            
            # Post-processing to fix placeholder tags
            for j in range(len(batch_preds)):
                batch_preds[j] = restore_masked_numbers(batch_src[j], batch_preds[j])
                
            predictions.extend(batch_preds)
            
        # 🕵️ DIAGNOSTIC BLOCK
        print("\n" + "="*70)
        print(f"🕵️ ABLATION SAMPLES FOR ({src_col} -> {TARGET_LANG}) 🕵️")
        print("="*70)

        sample_indices = random.sample(range(len(predictions)), min(5, len(predictions)))

        for idx, rand_i in enumerate(sample_indices):
            print(f"\n--- Sample {idx+1} ---")
            print(f"Source ({src_col}):      {src_texts[rand_i]}")
            print(f"Reference ({TARGET_LANG}):   {ref_texts[rand_i]}")
            print(f"Prediction ({TARGET_LANG}):  {predictions[rand_i]}")

        print("\n" + "="*70)
            
        # Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 5. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR ABLATION (No Contextual Encoder): {TARGET_LANG}")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    # Save with a specific ablation filename
    results_df.to_csv(f"ablation_no_context_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Ablation results saved to ablation_no_context_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

Using device: cuda
Loading Custom Pipeline Weights...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading Global Metric Evaluators...


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.decoder.bias         | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 
predictions.decoder.weight       | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Searching for pairs ending in ..._to_or.csv

[ABLATION] Testing Pair: as -> or (Bypassing Contextual Encoder)

🕵️ ABLATION SAMPLES FOR (as -> or) 🕵️

--- Sample 1 ---
Source (as):      বিভিন্ন ভূ-আৱৰণৰ শ্রেণীসমূহ চিনাক্ত কৰিবৰ বাবে বৰ্ণমাত্ৰা, বিন্যাস, আকৃতি, আর্হি আৰু আন আন বস্তুৰ সৈতে থকা সম্পর্কৰ নিচিনা দৃশ্যমান সংকেতৰ ব্যৱহাৰ কৰাটো বর্ণনাকাৰীগৰাকীৰ ওপৰত নির্ভৰ কৰে।
Reference (or):   ଏହା ବିଭିନ୍ନ ଭୂ-ଆବରଣ ଶ୍ରେଣୀର ଚିହ୍ନଟ କରିବା ପାଇଁ ବର୍ଣ୍ଣ, ବିନ୍ୟାସ, ଆକୃତି, ରୂପରେଖ, ଏବଂ ଅନ୍ୟାନ୍ୟ ବସ୍ତୁଗୁଡ଼ିକ ସହିତ ଏହାର ସମ୍ପର୍କ ଭଳି ଚାକ୍ଷୁଷ ସୂତ୍ରଗୁଡ଼ିକୁ ନିୟୋଜିତ କରିବାକୁ ବ୍ୟାଖ୍ୟାକାରଜଣକଙ୍କ ଉପରେ ନିର୍ଭର କରିଥାଏ ।
Prediction (or):  ଯେଉଁମାନେ ସେମାନଙ୍କ ପାଖରେ ଥିଲେ, ସେମାନେ ଯେଉଁମାନେ ସେମାନଙ୍କ ପାଖରେ ରହୁଥିଲେ, ସେମାନେ ସେମାନଙ୍କ 

--- Sample 2 ---
Source (as):      এইটো কৰিবলৈ তেওঁলোকে কেনেকৈ সফলভাৱে ড্রিবল কৰি আৰু খেলুৱৈসকলৰ মাজেৰে বলটো একেবাৰে সঠিকভাৱে আৰু খৰতকীয়াকৈ লৈ গৈ প'জেছন ৰাখিব লাগে সেইটো শিকাৰ প্রয়োজন।
Reference (or):   ଏହା କରିବା ପାଇଁ, ସେମାନଙ୍କୁ କିପରି ଫଳପ୍ରଦ ଭାବରେ ବଲ୍‌ଟିକୁ ଗଡ଼ାଇ ଅକ୍ତିଆରରେ ରଖିବାକୁ ହେବ ଏବଂ ବଲ୍‌ଟିକୁ ଖେଳାଳି